# Training Basics: solutions

**Author:** Nassim Razaaly ([nassim.razaaly@ensma.fr](mailto:nassim.razaaly@ensma.fr))

Solutions of the exercises of the notebook *Training Basics*. The MLP, the training loop and the image tools are in `x_lib_training.py`, loaded by the first cell. The numbers quoted in the text are indicative: yours may differ a little (random initialization, CPU/GPU).

In [ ]:
# Download the course library (once) and import its tools
import os, urllib.request
LIB = "x_lib_training.py"
if not os.path.exists(LIB):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/nrazaaly/coursePINNs/main/notebooks/{LIB}", LIB)
from x_lib_training import *
from matplotlib import cbook
print("device:", device)

## 1. A 1D function

Tools of this part: the function $f$, the data (same as in the main notebook), a training with Adam and an optional cosine schedule, the MSE in the original units of $y$, and a plot of the predictions.

In [ ]:
def f(x):
    return torch.sin(3 * x) + 0.3 * x**2 - 1.5 * torch.exp(-10 * (x - 1) ** 2)

def generate_data(N, noise, seed=0):
    # noisy samples of f on [-3, 3], split 60 / 20 / 20 %, normalized with the training statistics
    g = torch.Generator().manual_seed(seed)
    x = torch.rand(N, 1, generator=g) * 6 - 3
    y = f(x) + noise * torch.randn(N, 1, generator=g)
    perm = torch.randperm(N, generator=g)
    idx = {"train": perm[:int(0.6 * N)], "val": perm[int(0.6 * N):int(0.8 * N)], "test": perm[int(0.8 * N):]}
    tr = idx["train"]
    norm = {"x": (x[tr].mean(), x[tr].std()), "y": (y[tr].mean(), y[tr].std())}
    data = {k: (((x[i] - norm["x"][0]) / norm["x"][1]).to(device), ((y[i] - norm["y"][0]) / norm["y"][1]).to(device))
            for k, i in idx.items()}
    return data, norm

def run(model, data, lr, n_epochs, n_batch=1, cosine=True, plot_every=None, **kw):
    # Adam, with a cosine schedule (lr -> 0 at the last epoch) if cosine=True
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs) if cosine else None
    return train(model.to(device), optimizer, data, n_epochs, n_batch, scheduler=scheduler,
                 plot_every=plot_every or max(1, n_epochs // 10), **kw)

def mse(model, data, norm, name):
    # MSE on one set, in the original units of y
    with torch.no_grad():
        return loss_fn(model(data[name][0]), data[name][1]).item() * norm["y"][1].item() ** 2

def plot_fits(models, data, norm, title=""):
    x = torch.linspace(-3, 3, 500)[:, None]
    X, Y = data["train"]
    plt.figure(figsize=(8, 3.5))
    plt.scatter(norm["x"][0] + norm["x"][1] * X.cpu(), norm["y"][0] + norm["y"][1] * Y.cpu(), s=10, c="gray", label="training points")
    plt.plot(x, f(x), "k", lw=2, label="true function")
    for label, m in models.items():
        with torch.no_grad():
            y = m(((x - norm["x"][0]) / norm["x"][1]).to(device)).cpu()
        plt.plot(x, norm["y"][0] + norm["y"][1] * y, label=label)
    plt.title(title); plt.legend(fontsize=8); plt.show()

### Exercise 1

**Best settings:** 2 hidden layers of 64 neurons, `nn.Tanh`, Adam with `lr = 1e-2` and a **cosine schedule** (`lr` decreases from `1e-2` to 0 along the training), full batch (`n_batch = 1`), 2000 epochs. Compared with the settings of section 1.2, on the same data ($N = 100$, `noise = 0.2`). Reference: a perfect model has an MSE of about `noise`$^2 = 0.04$ (the MSE is measured on noisy values).

In [ ]:
data, norm = generate_data(N=100, noise=0.2)

torch.manual_seed(0)
model_12 = MLP([1, 32, 1], nn.Tanh)
run(model_12, data, lr=1e-3, n_epochs=2000, cosine=False)

torch.manual_seed(0)
model_best = MLP([1, 64, 64, 1], nn.Tanh)
run(model_best, data, lr=1e-2, n_epochs=2000)

for label, m in [("section 1.2 settings", model_12), ("best settings", model_best)]:
    print(f"{label:20s}: train MSE = {mse(m, data, norm, 'train'):.4f} | val MSE = {mse(m, data, norm, 'val'):.4f}")
plot_fits({"section 1.2 settings": model_12, "best settings": model_best}, data, norm, "N = 100, noise = 0.2")

* The section 1.2 model is **under-trained**: `lr = 1e-3` is too small, the loss is still decreasing after 2000 epochs and the curve misses the dip at $x = 1$.
* Best settings: 2 layers of 64 neurons are expressive enough for $f$; Tanh is smooth like $f$; a large `lr` makes fast progress, and the cosine schedule ends with small steps, so the loss converges without oscillating; with 60 points, full batch is fast and exact. The validation MSE is divided by 4.
* The remaining error comes from the **data**: 60 noisy points (and the validation MSE on 20 points is itself noisy).

**More data** ($N = 1000$, same settings): our final model, its test MSE is computed once.

In [ ]:
data_1000, norm_1000 = generate_data(N=1000, noise=0.2)
torch.manual_seed(0)
model_1000 = MLP([1, 64, 64, 1], nn.Tanh)
run(model_1000, data_1000, lr=1e-2, n_epochs=2000)

print(f"N = 1000: val MSE = {mse(model_1000, data_1000, norm_1000, 'val'):.4f} | test MSE = {mse(model_1000, data_1000, norm_1000, 'test'):.4f}")
plot_fits({"best settings, N = 1000": model_1000}, data_1000, norm_1000, "N = 1000, noise = 0.2")

Validation and test MSE close to `noise`$^2 = 0.04$: the model cannot do better, and its curve lies on the true function.

### Exercise 2

Same settings with `noise = 0.01`, for $N = 100$ and $N = 1000$. With little noise the loss can go much lower, so the training is longer: 5000 epochs.

In [ ]:
data_01, norm_01 = generate_data(N=100, noise=0.01)
torch.manual_seed(0)
model_01 = MLP([1, 64, 64, 1], nn.Tanh)
run(model_01, data_01, lr=1e-2, n_epochs=5000)

data_01_1000, norm_01_1000 = generate_data(N=1000, noise=0.01)
torch.manual_seed(0)
model_01_1000 = MLP([1, 64, 64, 1], nn.Tanh)
run(model_01_1000, data_01_1000, lr=1e-2, n_epochs=5000)

for label, m, d, nm, noise in [("noise = 0.2,  N = 100 ", model_best, data, norm, 0.2), ("noise = 0.2,  N = 1000", model_1000, data_1000, norm_1000, 0.2),
                               ("noise = 0.01, N = 100 ", model_01, data_01, norm_01, 0.01), ("noise = 0.01, N = 1000", model_01_1000, data_01_1000, norm_01_1000, 0.01)]:
    print(f"{label}: val MSE = {mse(m, d, nm, 'val'):.1e} | noise^2 = {noise ** 2:.0e}")
plot_fits({"best settings": model_01}, data_01, norm_01, "N = 100, noise = 0.01")

* Even a perfect model has an MSE of about `noise`$^2$: the noise sets the lowest reachable value.
* `noise = 0.2`: the MSE is close to `noise`$^2$ (2 times with 60 points, at the noise level with 600): the noise dominates, a better model would hardly change the MSE.
* `noise = 0.01`: with 60 points the MSE is about 10 times `noise`$^2$: the error now comes from the **model and the data density** (the points are too sparse to pin down the narrow dip at $x = 1$). With 600 points it reaches `noise`$^2$.
* Conclusion: with noisy data the noise dominates the MSE; with clean data the hyperparameters and the amount of data make all the difference.

### Exercise 3

A large model (3 hidden layers of 128 neurons, SiLU), few points ($N = 50$: 30 training points), a long training (4000 epochs, constant `lr = 1e-3`).

In [ ]:
data_50, norm_50 = generate_data(N=50, noise=0.2)
torch.manual_seed(0)
model_over = MLP([1, 128, 128, 128, 1], nn.SiLU)
hist_over = run(model_over, data_50, lr=1e-3, n_epochs=4000, cosine=False)
print(f"train MSE = {mse(model_over, data_50, norm_50, 'train'):.4f} | val MSE = {mse(model_over, data_50, norm_50, 'val'):.4f} "
      f"| lowest val MSE at epoch {hist_over['best_epoch']}")

The training MSE goes far below `noise`$^2 = 0.04$: the model learns the **noise** of the 30 points. The validation MSE reaches its minimum after about 500 epochs, then increases.

Two remedies, with the same model:

* **early stopping:** stop the training at the epoch with the lowest validation MSE (same seed, so the same training, stopped at `best_epoch`);
* **weight decay:** `torch.optim.AdamW` with `weight_decay = 1`, which pulls the weights towards 0 and makes the function smoother.

In [ ]:
torch.manual_seed(0)
model_early = MLP([1, 128, 128, 128, 1], nn.SiLU)
run(model_early, data_50, lr=1e-3, n_epochs=hist_over["best_epoch"], cosine=False)

torch.manual_seed(0)
model_wd = MLP([1, 128, 128, 128, 1], nn.SiLU).to(device)
optimizer = torch.optim.AdamW(model_wd.parameters(), lr=1e-3, weight_decay=1.0)
train(model_wd, optimizer, data_50, n_epochs=4000, n_batch=1, plot_every=400)

models = {"overfitted": model_over, "early stopping": model_early, "weight decay": model_wd}
for label, m in models.items():
    print(f"{label:15s}: train MSE = {mse(m, data_50, norm_50, 'train'):.4f} | val MSE = {mse(m, data_50, norm_50, 'val'):.4f}")
plot_fits(models, data_50, norm_50, "Overfitting and remedies (N = 50)")

* **Overfitted:** the curve bends towards the noisy training points.
* **Early stopping** and **weight decay**: smoother curves, closer to $f$, and a lower validation MSE (early stopping is free: it is the same training, stopped earlier).
* Other remedies: more data (exercise 1), a smaller model.

### Exercise 4

Data of exercise 2 (`noise = 0.01`, $N = 100$), where the model error is not hidden by the noise; same settings, only the activation changes.

In [ ]:
models_act = {}
for act in [nn.ReLU, nn.Tanh, nn.SiLU]:
    torch.manual_seed(0)
    models_act[act.__name__] = MLP([1, 64, 64, 1], act)
    run(models_act[act.__name__], data_01, lr=1e-2, n_epochs=5000)
    print(f"{act.__name__:5s}: val MSE = {mse(models_act[act.__name__], data_01, norm_01, 'val'):.1e}")
plot_fits(models_act, data_01, norm_01, "Activation functions (noise = 0.01, N = 100)")

Tanh and SiLU are better than ReLU (about 2 times lower validation MSE). With ReLU the MLP is **piecewise linear**: it approximates the smooth $f$ by straight segments with kinks, and needs many neurons for the curved parts. Tanh and SiLU are smooth, like $f$. (With `noise = 0.2`, the three are close: the noise dominates.)

### To go further: learning rate schedules

The three schedulers of the main notebook, and a constant `lr`, on clean data ($N = 1000$, `noise = 0.01`), `lr = 1e-2` at the start, 2000 epochs.

In [ ]:
schedulers = {"constant": lambda opt: None,
              "cosine": lambda opt: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=2000, eta_min=1e-7),
              "step": lambda opt: torch.optim.lr_scheduler.StepLR(opt, step_size=500, gamma=0.5),
              "exponential": lambda opt: torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.999)}
hists_sched = {}
for name, make in schedulers.items():
    torch.manual_seed(0)
    m = MLP([1, 64, 64, 1], nn.Tanh).to(device)
    optimizer = torch.optim.Adam(m.parameters(), lr=1e-2)
    hists_sched[name] = train(m, optimizer, data_01_1000, n_epochs=2000, n_batch=1, plot_every=200, scheduler=make(optimizer))

plt.figure(figsize=(7, 3.5))
for name, h in hists_sched.items():
    plt.semilogy(h["val"], lw=1, label=name)
plt.xlabel("epoch"); plt.ylabel("validation MSE (normalized)"); plt.legend(); plt.show()

* **Constant `lr`:** fast progress, but the loss keeps jumping (spikes of a factor 10 to 100): the final model depends on the epoch where we stop.
* **With a schedule:** large steps first, small ones at the end: the spikes disappear and the loss converges smoothly. The three schedules end at a similar value here; cosine has a single parameter (`T_max = n_epochs`), our default choice.

### To go further: a 2D function

$f_2(x_1, x_2) = \sin(\pi x_1)\cos(\pi x_2)$ on $[-1, 1]^2$, $N = 300$ noisy points (`noise = 0.01`). Inputs in $[-1, 1]$ and outputs of order 1: no normalization needed. Model: 2 hidden layers of 32 neurons, Tanh, `lr = 1e-2` with cosine schedule, 3000 epochs.

In [ ]:
def f2(x):                                                    # (N, 2) -> (N, 1)
    return torch.sin(torch.pi * x[:, :1]) * torch.cos(torch.pi * x[:, 1:])

g = torch.Generator().manual_seed(0)
x2 = torch.rand(300, 2, generator=g) * 2 - 1
y2 = f2(x2) + 0.01 * torch.randn(300, 1, generator=g)
data_2d = {"train": (x2[:180].to(device), y2[:180].to(device)), "val": (x2[180:240].to(device), y2[180:240].to(device)),
           "test": (x2[240:].to(device), y2[240:].to(device))}

torch.manual_seed(0)
model_2d = MLP([2, 32, 32, 1], nn.Tanh)
hist_2d = run(model_2d, data_2d, lr=1e-2, n_epochs=3000)
print(f"train MSE = {hist_2d['train'][-1]:.1e} | val MSE = {hist_2d['val'][-1]:.1e} | noise^2 = 1e-04")

t = torch.linspace(-1, 1, 100)
grid = torch.stack([m.flatten() for m in torch.meshgrid(t, t, indexing="ij")], dim=1)
with torch.no_grad():
    pred = model_2d(grid.to(device)).cpu().reshape(100, 100)
true = f2(grid).reshape(100, 100)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4), layout="constrained")
for a, z, title in zip(ax, [true, pred, pred - true], ["true $f_2$", "MLP", "error (MLP - true)"]):
    im = a.imshow(z, extent=[-1, 1, -1, 1], origin="lower", cmap="RdBu", vmin=-1, vmax=1)
    a.set(title=title, xlabel="$x_2$", ylabel="$x_1$")
fig.colorbar(im, ax=ax, label="same scale for the 3 maps"); plt.show()

With 180 points, a small MLP learns $f_2$ in a few seconds: validation MSE of a few $10^{-4}$, and the error map is almost blank at the scale of the function.

## 2. Images

The two images of the main notebook, and their pixel data (same positions for both). `run` (part 1) is used for the trainings.

In [ ]:
img = load_image(cbook.get_sample_data("Minduka_Present_Blue_Pack.png", asfileobj=False), gray=True)
img_color = load_image(cbook.get_sample_data("grace_hopper.jpg", asfileobj=False), size=128)
data_img, X_all = image_data(img)
data_color, X_all = image_data(img_color)
show_images([img.numpy(), img_color.numpy()], ["drawing (gray)", "portrait (color)"])

## 3. Fourier features (section 2.7)

`MLP_FF(2, n_freq, sigma, layers, ...)`: Fourier features ($2 \to$ `2 * n_freq`), then an MLP. Exercises 1 to 4 use short trainings (constant `lr = 1e-3`, 100 mini-batches per epoch) to compare settings quickly.

### Exercise 1

Same MLP $[128, 128, 1]$ with and without Fourier features (`n_freq = 128`, $\sigma = 10$, as in section 2.7), 30 epochs.

In [ ]:
torch.manual_seed(0)
models_ff = {"no Fourier features": MLP([2, 128, 128, 1], nn.ReLU, nn.Sigmoid),
             "Fourier features, sigma = 10": MLP_FF(2, 128, 10, [128, 128, 1], nn.ReLU, nn.Sigmoid)}
hists_ff = {}
for label, m in models_ff.items():
    hists_ff[label] = run(m, data_img, lr=1e-3, n_epochs=30, n_batch=100, cosine=False)
show_images([img.numpy()] + [predict_image(m, X_all, img.shape) for m in models_ff.values()],
            ["original"] + [f"{k}\nval {psnr(hists_ff[k]['val'][-1]):.1f} dB" for k in models_ff], size=4)

* Without Fourier features: about 18 dB, blurry.
* With Fourier features: the edges are **sharp** in the same number of epochs, but the image is noisy and the validation PSNR only about 20 dB: with $\sigma = 10$ the frequencies are too high, the model oscillates between the training pixels. $\sigma$ must be chosen (exercise 2).

### Exercises 2 and 3




**Final model**, trained to the end: Fourier features with `n_freq = 256` and $\sigma = 1.5$ (with a long training, a slightly smaller $\sigma$ is best), MLP $[128, 128, 1]$, 100 epochs, `lr = 1e-3` with cosine schedule. Its test PSNR is computed once.

In [ ]:
# Initialize and train
torch.manual_seed(0)
model_ff = MLP_FF(2, 256, 1.5, [128, 128, 1], nn.ReLU, nn.Sigmoid)
print(model_ff)
model_ff.show()
hist_ff = run(model_ff, data_img, lr=1e-3, n_epochs=100, n_batch=100)


In [ ]:
# Show model capability
with torch.no_grad():
    test_psnr = psnr(loss_fn(model_ff(data_img["test"][0]), data_img["test"][1]).item())
print(f"train PSNR = {psnr(hist_ff['train'][-1]):.1f} dB | val PSNR = {psnr(hist_ff['val'][-1]):.1f} dB | "
      f"test PSNR = {test_psnr:.1f} dB | training time {hist_ff['time']:.0f} s")
show_images([img.numpy(), predict_image(model_ff, X_all, img.shape)],
            ["original", f"Fourier features: val {psnr(hist_ff['val'][-1]):.1f} dB"], size=4)

About **31 dB** on the validation pixels and 32 dB on the test pixels (against 25 dB for the best plain MLP), with fewer parameters and a shorter training: differences with the original are hard to see. The training PSNR is much higher (about 40 dB): the remaining error comes from the 40 % of pixels the model must guess.




### Exercise 5

Color portrait with Fourier features: `n_freq = 256`, $\sigma = 2$, MLP $[256, 256, 3]$, same training as the final model of exercise 2 (about 2 min on a Colab CPU).

In [ ]:
torch.manual_seed(0)
model_ff_col = MLP_FF(2, 256, 2, [256, 256, 3], nn.ReLU, nn.Sigmoid)
hist_ff_col = run(model_ff_col, data_color, lr=1e-3, n_epochs=100, n_batch=100)


In [ ]:
print(f"val PSNR = {psnr(hist_ff_col['val'][-1]):.1f} dB")
show_images([img_color.numpy(), predict_image(model_ff_col, X_all, img_color.shape)],
            ["original", f"Fourier features: val {psnr(hist_ff_col['val'][-1]):.1f} dB"], size=4)

About **26 dB**, against 20 dB without Fourier features: the face, the glasses and the uniform are sharp. The fine textures (hair, medals, JPEG noise) cannot be predicted from the neighboring pixels: the limit comes from the 40 % of unknown pixels, not from the model. With more training pixels (e.g. `image_data(img_color, train=0.9, val=0.05)`), the PSNR on the remaining pixels increases.